# ReNewGenie: detecting several waste items in one photo (YOLO)

A normal classifier answers "what is this photo?". Real waste piles hold many items, so this notebook trains an **object detector** that draws a box around each item and names it.

Run in Colab with a **T4 GPU**. You need a free **Roboflow** account and API key (Roboflow Universe hosts public waste-detection datasets; the TACO dataset is a common choice). Pick a dataset, choose export format **YOLOv8**, and paste the workspace, project and version below.

> Written without a GPU, so it is untested. Expect to adjust dataset names. Report the mAP numbers it prints, nothing else.

In [ ]:
!pip -q install ultralytics roboflow
import os
from roboflow import Roboflow
ROBOFLOW_KEY = ""   # paste your key (Roboflow > Settings > API Key). Do not share this notebook with the key in it.
WORKSPACE, PROJECT, VERSION = "", "", 1   # e.g. from the dataset's Universe page, "Download Dataset" > YOLOv8 > show download code
assert ROBOFLOW_KEY and WORKSPACE and PROJECT, "fill in the three values above"
rf = Roboflow(api_key=ROBOFLOW_KEY)
ds = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")
print(ds.location)

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8s.pt")      # small model pretrained on COCO; use yolov8n.pt if Colab runs out of time
res = model.train(data=os.path.join(ds.location, "data.yaml"), epochs=60, imgsz=640, batch=16, patience=15, seed=7)

In [ ]:
# Honest evaluation on the dataset's test split
m = model.val(split="test")
print("mAP@0.5     : %.3f" % m.box.map50)
print("mAP@0.5:0.95: %.3f" % m.box.map)
for name, ap in zip(model.names.values(), m.box.maps):
    print("%-20s AP %.3f" % (name, ap))

In [ ]:
# Try it on your own photo of a pile of waste
from google.colab import files as colab_files
up = colab_files.upload(); path = list(up)[0]
r = model.predict(path, conf=0.25)[0]
r.save("detected.jpg")
from IPython.display import Image; display(Image("detected.jpg"))
for b in r.boxes: print(model.names[int(b.cls)], "%.2f" % float(b.conf))

In [ ]:
# Export for the browser (TensorFlow.js) or phone (TFLite)
model.export(format="tfjs")      # or format="tflite"
!zip -qr yolo_export.zip runs
colab_files.download("yolo_export.zip")

## Notes
- mAP@0.5 above about 0.5 is decent for waste datasets; these are hard (crumpled, overlapping, many tiny classes).
- Merge rare classes into the six used by the demo (Cardboard, Glass, Metal, Paper, Plastic, Trash) if the dataset has dozens of labels. That makes both training and the report simpler.
- Running a detector in the live demo page needs the exported TensorFlow.js model hosted next to the page, so it belongs in the GitHub repo, not in a single-file artifact.